In [1]:
!pip install -q ultralytics huggingface_hub transformers fastapi uvicorn pyngrok pillow nest_asyncio python-multipart rasterio mercantile


In [ ]:
!rm -f ngrok ngrok.zip ngrok-stable-linux-amd64.tgz
!wget -O ngrok-stable-linux-amd64.tgz https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar -xvzf ngrok-stable-linux-amd64.tgz
!chmod +x ngrok
!./ngrok version
!./ngrok authtoken  "YOUR TOKEN HERE"


In [ ]:
import os, io, json, cv2, torch, requests, nest_asyncio, uvicorn, mercantile
import xml.etree.ElementTree as ET
from typing import Optional
from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field
from pyngrok import ngrok
from PIL import Image
import numpy as np
from ultralytics import YOLO
from huggingface_hub import hf_hub_download
from transformers import AutoImageProcessor, AutoModelForSemanticSegmentation

nest_asyncio.apply()

# ==========================================================
# 1. Basemap Configuration (Multi-Basemap Support)
# ==========================================================
BASEMAPS = {
    "nlsc_uav": {
        "name": "Taiwan NLSC UAV Imagery",
        "url": "https://wmts.nlsc.gov.tw/wmts/UAV111A03/default/GoogleMapsCompatible/{z}/{y}/{x}.png",
        "max_zoom": 20,
        "detect_zoom": 20,
        "attribution": "Taiwan NLSC UAV WMTS"
    },
    "nlsc_photo2": {
        "name": "Taiwan NLSC Ortho Photo (PHOTO2)",
        "url": "https://wmts.nlsc.gov.tw/wmts/PHOTO2/default/GoogleMapsCompatible/{z}/{y}/{x}",
        "max_zoom": 20,
        "detect_zoom": 19,
        "attribution": "Taiwan NLSC PHOTO2"
    },
    "nlsc_photo_mix": {
        "name": "Taiwan NLSC Ortho + Labels (PHOTO_MIX)",
        "url": "https://wmts.nlsc.gov.tw/wmts/PHOTO_MIX/default/GoogleMapsCompatible/{z}/{y}/{x}",
        "max_zoom": 20,
        "detect_zoom": 19,
        "attribution": "Taiwan NLSC PHOTO_MIX"
    },
    "nlsc_emap": {
        "name": "Taiwan NLSC e-Map",
        "url": "https://wmts.nlsc.gov.tw/wmts/EMAP/default/GoogleMapsCompatible/{z}/{y}/{x}",
        "max_zoom": 20,
        "detect_zoom": 19,
        "attribution": "Taiwan NLSC EMAP"
    },
    "esri_satellite": {
        "name": "Esri World Imagery",
        "url": "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        "max_zoom": 19,
        "detect_zoom": 19,
        "attribution": "Tiles &copy; Esri"
    },
    "google_satellite": {
        "name": "Google Satellite",
        "url": "https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
        "max_zoom": 20,
        "detect_zoom": 20,
        "attribution": "Google"
    },
    "osm": {
        "name": "OpenStreetMap",
        "url": "https://tile.openstreetmap.org/{z}/{x}/{y}.png",
        "max_zoom": 19,
        "detect_zoom": 19,
        "attribution": "&copy; OpenStreetMap contributors"
    }
}

def load_nlsc_uav_wmts_layers():
    """Auto-discover all available NLSC UAV WMTS layers & their WGS84 bounding boxes from GetCapabilities."""
    endpoints = [
        (
            "https://wmts.nlsc.gov.tw/wmts/1.0.0/WMTSCapabilities.xml",
            "https://wmts.nlsc.gov.tw/wmts/{layer}/default/GoogleMapsCompatible/{{z}}/{{y}}/{{x}}.png"
        ),
        (
            "https://maps.nlsc.gov.tw/wmtsUAS/wmts?SERVICE=WMTS&REQUEST=GetCapabilities&VERSION=1.0.0",
            "https://maps.nlsc.gov.tw/wmtsUAS/{layer}/default/GoogleMapsCompatible/{{z}}/{{y}}/{{x}}.png"
        ),
    ]
    added = 0
    for cap_url, tile_tpl in endpoints:
        try:
            resp = requests.get(cap_url, headers={"User-Agent": "UAV-Detection-App/2.0"}, timeout=15)
            resp.raise_for_status()
            root = ET.fromstring(resp.content)
            for layer_el in root.iter():
                if not layer_el.tag.endswith("}Layer") and layer_el.tag != "Layer":
                    continue
                ident, title, bounds = None, None, None
                for child in layer_el:
                    tag = child.tag.split("}")[-1]
                    if tag == "Identifier" and child.text:
                        ident = child.text.strip()
                    elif tag == "Title" and child.text:
                        title = child.text.strip()
                    elif tag == "WGS84BoundingBox":
                        lc, uc = None, None
                        for b in child:
                            btag = b.tag.split("}")[-1]
                            if btag == "LowerCorner" and b.text:
                                lc = [float(v) for v in b.text.strip().split()]
                            elif btag == "UpperCorner" and b.text:
                                uc = [float(v) for v in b.text.strip().split()]
                        if lc and uc and len(lc) == 2 and len(uc) == 2:
                            # Leaflet bounds format: [[sw_lat, sw_lon], [ne_lat, ne_lon]]
                            bounds = [[lc[1], lc[0]], [uc[1], uc[0]]]
                if not ident:
                    continue
                if ("wmtsUAS" in cap_url) or ident.upper().startswith("UAV"):
                    key = f"nlsc_{ident.lower()}"
                    if key == "nlsc_uav111a03":
                        if bounds:
                            BASEMAPS["nlsc_uav"]["bounds"] = bounds
                        if title:
                            BASEMAPS["nlsc_uav"]["name"] = f"[NLSC UAV] {title} ({ident})"
                        continue
                    if key not in BASEMAPS:
                        BASEMAPS[key] = {
                            "name": f"[NLSC UAV] {title or ident} ({ident})",
                            "url": tile_tpl.format(layer=ident),
                            "max_zoom": 20,
                            "detect_zoom": 20,
                            "attribution": f"Taiwan NLSC {ident}",
                            "bounds": bounds
                        }
                        added += 1
        except Exception as e:
            print(f"Note: Could not fetch WMTS capabilities from {cap_url}: {e}")
    print(f"Loaded {added} additional NLSC UAV WMTS layers (total basemaps: {len(BASEMAPS)}).")

load_nlsc_uav_wmts_layers()

# ==========================================================
# 2. Model Registry (Satellite & UAV Ortho Imagery — DOTAv1 OBB)
#    All models are trained specifically for top-down aerial / satellite
#    ortho imagery (DOTAv1) and auto-download if not in /content/.
# ==========================================================
MODEL_REGISTRY = {
    "vehicles_obb_x": {
        "name": "Vehicles — High Accuracy (YOLOv11x-OBB)",
        "weights": "yolo11x-obb.pt",
        "custom_weights": "/content/yolo11x-obb.pt",
        "task": "obb",
        "allowed_classes": ["small vehicle", "large vehicle"],
        "description": "High-accuracy small & large vehicle OBB detection on UAV/satellite ortho imagery."
    },
    "vehicles_obb_m": {
        "name": "Vehicles — Fast (YOLOv11m-OBB)",
        "weights": "yolo11m-obb.pt",
        "custom_weights": "/content/yolo11m-obb.pt",
        "task": "obb",
        "allowed_classes": ["small vehicle", "large vehicle"],
        "description": "Faster medium-size OBB model for large-area UAV/satellite vehicle detection."
    },
    "dota_all_obb": {
        "name": "All 15 Aerial Classes (DOTAv1 YOLOv11x-OBB)",
        "weights": "yolo11x-obb.pt",
        "custom_weights": "/content/yolo11x-obb.pt",
        "task": "obb",
        "allowed_classes": None,
        "description": "Detects all 15 DOTAv1 top-down satellite/UAV classes."
    },
    "maritime_aviation_obb": {
        "name": "Ships, Harbors & Aircraft (YOLOv11x-OBB)",
        "weights": "yolo11x-obb.pt",
        "custom_weights": "/content/yolo11x-obb.pt",
        "task": "obb",
        "allowed_classes": ["ship", "harbor", "plane", "helicopter"],
        "description": "Top-down detection of ships, harbors, planes, and helicopters."
    },
    "infrastructure_obb": {
        "name": "Bridges, Tanks & Roundabouts (YOLOv11x-OBB)",
        "weights": "yolo11x-obb.pt",
        "custom_weights": "/content/yolo11x-obb.pt",
        "task": "obb",
        "allowed_classes": ["bridge", "storage tank", "roundabout"],
        "description": "Top-down detection of bridges, storage tanks, and roundabouts."
    },
    "sports_facilities_obb": {
        "name": "Sports Courts & Pools (YOLOv11x-OBB)",
        "weights": "yolo11x-obb.pt",
        "custom_weights": "/content/yolo11x-obb.pt",
        "task": "obb",
        "allowed_classes": ["baseball diamond", "tennis court", "basketball court", "ground track field", "soccer ball field", "swimming pool"],
        "description": "Top-down detection of sports fields, courts, and swimming pools."
    },
    "buildings_yolo_seg": {
        "name": "Buildings Footprint (YOLOv8m-Seg Satellite)",
        "backend": "yolo_hf",
        "hf_repo": "keremberke/yolov8m-building-segmentation",
        "hf_filename": "best.pt",
        "task": "segment",
        "allowed_classes": None,
        "description": "Satellite/aerial building footprint instance segmentation (auto-downloads from Hugging Face)."
    },
    "roads_segformer": {
        "name": "Roads / Impervious (SegFormer-B3 Ortho)",
        "backend": "segformer",
        "hf_repo": "geospaitial-lab/segformer-b3-potsdam-512-512",
        "task": "semantic",
        "class_keywords": ["impervious", "road", "pavement", "street"],
        "description": "High-resolution aerial ortho road / impervious surface segmentation (ISPRS Potsdam)."
    },
    "buildings_roads_segformer": {
        "name": "Buildings + Roads + Land Cover (SegFormer-B3 Ortho)",
        "backend": "segformer",
        "hf_repo": "geospaitial-lab/segformer-b3-potsdam-512-512",
        "task": "semantic",
        "exclude_keywords": ["clutter", "background"],
        "description": "High-resolution aerial ortho semantic segmentation: buildings, roads, trees, vegetation & cars."
    }
}

DEFAULT_BASEMAP = "nlsc_uav"
DEFAULT_MODEL = "vehicles_obb_x"
IOU_THRESH = 0.5
TILE_SIZE = 256
HTTP_HEADERS = {"User-Agent": "UAV-Detection-App/2.0"}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Cache loaded models so shared weights load only once
LOADED_MODELS = {}

def resolve_model_spec(cfg: dict):
    backend = cfg.get("backend", "yolo")
    if backend == "yolo":
        custom_path = cfg.get("custom_weights")
        if custom_path and os.path.exists(custom_path):
            return backend, custom_path, cfg["task"]
        return backend, cfg["weights"], cfg["task"]
    return backend, cfg["hf_repo"], cfg["task"]

def get_model(model_id: str):
    if model_id not in MODEL_REGISTRY:
        raise HTTPException(status_code=400, detail=f"Unknown model_id: {model_id}")
    cfg = MODEL_REGISTRY[model_id]
    backend, key, task = resolve_model_spec(cfg)
    cache_key = (backend, key)
    if cache_key not in LOADED_MODELS:
        if backend == "yolo":
            print(f"Loading aerial YOLO model: {key} ...")
            LOADED_MODELS[cache_key] = YOLO(key)
            print("Loaded classes:", LOADED_MODELS[cache_key].names)
        elif backend == "yolo_hf":
            print(f"Downloading {cfg['hf_repo']}/{cfg['hf_filename']} from Hugging Face ...")
            w_path = hf_hub_download(repo_id=cfg["hf_repo"], filename=cfg["hf_filename"])
            LOADED_MODELS[cache_key] = YOLO(w_path)
            print("Loaded classes:", LOADED_MODELS[cache_key].names)
        elif backend == "segformer":
            print(f"Loading SegFormer aerial ortho model: {key} on {DEVICE} ...")
            proc = AutoImageProcessor.from_pretrained(key)
            mdl = AutoModelForSemanticSegmentation.from_pretrained(key).to(DEVICE).eval()
            LOADED_MODELS[cache_key] = (proc, mdl)
            print("Loaded classes:", mdl.config.id2label)
    return LOADED_MODELS[cache_key], cfg, backend, task

# Preload default model (auto-downloads yolo11x-obb.pt if not in /content/)
get_model(DEFAULT_MODEL)

app = FastAPI(title="UAV Multi-Model & Multi-Basemap Detection API")

class DetectRequest(BaseModel):
    sw_lat: float
    sw_lon: float
    ne_lat: float
    ne_lon: float
    basemap: str = DEFAULT_BASEMAP
    model_id: str = DEFAULT_MODEL
    conf: float = Field(default=0.3, ge=0.05, le=0.95)

def xy_to_lonlat(x, y, tile, img_w=TILE_SIZE, img_h=TILE_SIZE):
    bounds = mercantile.bounds(tile)
    lon = bounds.west + (x / float(img_w)) * (bounds.east - bounds.west)
    lat = bounds.north - (y / float(img_h)) * (bounds.north - bounds.south)
    return lon, lat

def download_tile(tile, url_template: str, zoom: int):
    url = url_template.format(z=zoom, x=tile.x, y=tile.y)
    r = requests.get(url, headers=HTTP_HEADERS, timeout=15)
    r.raise_for_status()
    img = Image.open(io.BytesIO(r.content)).convert("RGB")
    return np.array(img), tile

def extract_polygons(results, model, task: str):
    """Extract (polygon_xy_points, label, score) tuples for obb, segment, or detect tasks."""
    detections = []
    if task == "obb" and getattr(results, "obb", None) is not None and len(results.obb) > 0:
        for poly, c, s in zip(results.obb.xyxyxyxy.tolist(), results.obb.cls.tolist(), results.obb.conf.tolist()):
            detections.append((poly, model.names[int(c)].lower(), float(s)))
    elif task == "segment" and getattr(results, "masks", None) is not None and results.masks is not None:
        for poly, c, s in zip(results.masks.xy, results.boxes.cls.tolist(), results.boxes.conf.tolist()):
            if len(poly) >= 3:
                detections.append((poly.tolist(), model.names[int(c)].lower(), float(s)))
    elif getattr(results, "boxes", None) is not None and len(results.boxes) > 0:
        for box, c, s in zip(results.boxes.xyxy.tolist(), results.boxes.cls.tolist(), results.boxes.conf.tolist()):
            x1, y1, x2, y2 = box
            poly = [[x1, y1], [x2, y1], [x2, y2], [x1, y2]]
            detections.append((poly, model.names[int(c)].lower(), float(s)))
    return detections

def extract_segformer_polygons(tile_np, proc_and_model, model_cfg: dict, conf: float):
    """Run Hugging Face SegFormer semantic segmentation and vectorize masks to polygons."""
    processor, model = proc_and_model
    img_h, img_w = tile_np.shape[:2]
    pil_img = Image.fromarray(tile_np)
    inputs = processor(images=pil_img, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = torch.nn.functional.interpolate(
        outputs.logits, size=(img_h, img_w), mode="bilinear", align_corners=False
    )
    probs = torch.softmax(logits, dim=1)[0]
    conf_map, pred_mask = probs.max(dim=0)
    pred_np = pred_mask.cpu().numpy()
    conf_np = conf_map.cpu().numpy()

    class_kw = model_cfg.get("class_keywords")
    excl_kw = model_cfg.get("exclude_keywords", [])
    detections = []

    for cls_id in np.unique(pred_np):
        raw_label = str(model.config.id2label.get(int(cls_id), str(cls_id))).lower()
        if any(k in raw_label for k in excl_kw):
            continue
        if class_kw is not None and not any(k in raw_label for k in class_kw):
            continue
        label = "road / impervious" if "impervious" in raw_label else raw_label
        bin_mask = ((pred_np == cls_id) & (conf_np >= conf)).astype(np.uint8)
        contours, _ = cv2.findContours(bin_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            if cv2.contourArea(cnt) < 40:
                continue
            approx = cv2.approxPolyDP(cnt, 1.5, True).reshape(-1, 2)
            if len(approx) < 3:
                continue
            c_mask = np.zeros((img_h, img_w), dtype=np.uint8)
            cv2.drawContours(c_mask, [cnt], -1, 1, thickness=-1)
            score = float(conf_np[c_mask == 1].mean()) if np.any(c_mask) else conf
            detections.append((approx.tolist(), label, score))
    return detections

@app.get("/config")
async def get_config():
    models_info = {}
    for mid, cfg in MODEL_REGISTRY.items():
        _, _, resolved_task = resolve_model_spec(cfg)
        models_info[mid] = {
            "name": cfg["name"],
            "task": resolved_task,
            "description": cfg["description"],
            "available": True
        }
    return {
        "basemaps": BASEMAPS,
        "models": models_info,
        "default_basemap": DEFAULT_BASEMAP,
        "default_model": DEFAULT_MODEL
    }

@app.post("/detect")
async def detect(req: DetectRequest):
    if req.basemap not in BASEMAPS:
        raise HTTPException(status_code=400, detail=f"Unknown basemap: {req.basemap}")

    basemap_cfg = BASEMAPS[req.basemap]
    zoom = basemap_cfg["detect_zoom"]
    model_obj, model_cfg, backend, task = get_model(req.model_id)
    allowed_classes = model_cfg.get("allowed_classes")

    tiles = list(mercantile.tiles(req.sw_lon, req.sw_lat, req.ne_lon, req.ne_lat, zoom))
    features = []

    for t in tiles:
        try:
            tile_np, _ = download_tile(t, basemap_cfg["url"], zoom)
        except Exception as e:
            print(f"Skipping tile {t}: {e}")
            continue

        img_h, img_w = tile_np.shape[:2]
        if backend == "segformer":
            raw_dets = extract_segformer_polygons(tile_np, model_obj, model_cfg, req.conf)
        else:
            results = model_obj.predict(
                tile_np,
                conf=req.conf,
                iou=IOU_THRESH,
                imgsz=max(img_h, img_w),
                task=task,
                verbose=False
            )[0]
            raw_dets = extract_polygons(results, model_obj, task)

        for poly, label, score in raw_dets:
            if allowed_classes is not None and label not in allowed_classes:
                continue

            coords_wgs84 = [xy_to_lonlat(x, y, t, img_w, img_h) for x, y in poly]
            lats = [lat for lon, lat in coords_wgs84]
            lons = [lon for lon, lat in coords_wgs84]

            # Filter detections inside user rectangle
            if min(lats) >= req.sw_lat and max(lats) <= req.ne_lat and \
               min(lons) >= req.sw_lon and max(lons) <= req.ne_lon:
                ring = [[lon, lat] for lon, lat in coords_wgs84]
                if ring[0] != ring[-1]:
                    ring.append(ring[0])  # Close GeoJSON polygon ring
                features.append({
                    "type": "Feature",
                    "properties": {
                        "label": label,
                        "score": round(score, 4),
                        "model": req.model_id,
                        "basemap": req.basemap
                    },
                    "geometry": {
                        "type": "Polygon",
                        "coordinates": [ring]
                    }
                })

    return {"type": "FeatureCollection", "features": features}

@app.get("/", response_class=HTMLResponse)
async def index():
    basemaps_json = json.dumps(BASEMAPS)
    models_meta = {
        mid: {
            "name": cfg["name"],
            "task": resolve_model_spec(cfg)[2],
            "description": cfg["description"],
            "available": True
        }
        for mid, cfg in MODEL_REGISTRY.items()
    }
    models_json = json.dumps(models_meta)

    html = f"""
<!doctype html>
<html>
<head>
<meta charset="utf-8"/>
<title>UAV Object Detection (Multi-Basemap & Multi-Model)</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<link rel="stylesheet" href="https://unpkg.com/leaflet-draw@1.0.4/dist/leaflet.draw.css"/>
<style>
html, body, #map {{ height: 100%; margin: 0; padding: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; }}
.panel {{ position: absolute; top: 10px; right: 10px; z-index: 1000; background: rgba(255,255,255,0.95); padding: 12px; border-radius: 8px; box-shadow: 0 2px 10px rgba(0,0,0,0.25); width: 250px; }}
.panel h4 {{ margin: 0 0 8px 0; font-size: 14px; color: #1e293b; }}
.field {{ margin-bottom: 8px; }}
.field label {{ display: block; font-size: 12px; font-weight: 600; color: #475569; margin-bottom: 3px; }}
.field select, .field input[type=range] {{ width: 100%; box-sizing: border-box; padding: 4px 6px; border-radius: 4px; border: 1px solid #cbd5e1; font-size: 12px; }}
.btn {{ display: block; width: 100%; text-align: center; background: #2563eb; color: #fff; padding: 7px; margin-top: 6px; cursor: pointer; border: none; border-radius: 5px; font-size: 13px; font-weight: 600; }}
.btn:hover {{ background: #1d4ed8; }}
.btn.secondary {{ background: #64748b; }}
.btn.secondary:hover {{ background: #475569; }}
.btn.success {{ background: #059669; }}
.btn.success:hover {{ background: #047857; }}
#status {{ margin-top: 8px; font-size: 12px; color: #334155; }}
#legend {{ margin-top: 8px; font-size: 11px; max-height: 150px; overflow-y: auto; border-top: 1px solid #e2e8f0; padding-top: 6px; }}
.legend-item {{ display: flex; align-items: center; gap: 6px; margin-bottom: 3px; }}
.swatch {{ width: 10px; height: 10px; border-radius: 2px; display: inline-block; }}
</style>
</head>
<body>
<div id="map"></div>
<div class="panel">
  <h4>UAV Object Detection</h4>
  <div class="field">
    <label for="basemapFilter">Basemap (NLSC UAV / Satellite)</label>
    <input id="basemapFilter" type="text" placeholder="Filter layers (e.g. UAV, 113, 台北)..." style="width:100%;box-sizing:border-box;padding:4px 6px;margin-bottom:4px;border-radius:4px;border:1px solid #cbd5e1;font-size:12px;"/>
    <select id="basemapSelect"></select>
  </div>
  <div class="field">
    <label for="modelSelect">Detection Model</label>
    <select id="modelSelect"></select>
  </div>
  <div class="field">
    <label for="confSlider">Confidence: <span id="confVal">0.30</span></label>
    <input id="confSlider" type="range" min="0.10" max="0.90" step="0.05" value="0.30"/>
  </div>
  <button class="btn" id="btnDetect">Detect Objects</button>
  <button class="btn success" id="btnDownload">Download GeoJSON</button>
  <button class="btn secondary" id="btnClear">Clear Detections</button>
  <div id="status">Draw a rectangle on the map to begin.</div>
  <div id="legend"></div>
</div>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://unpkg.com/leaflet-draw@1.0.4/dist/leaflet.draw.js"></script>
<script>
const BASEMAPS = {basemaps_json};
const MODELS = {models_json};
const DEFAULT_BASEMAP = '{DEFAULT_BASEMAP}';
const DEFAULT_MODEL = '{DEFAULT_MODEL}';

const CLASS_COLORS = {{
    'small vehicle': '#ef4444',
    'large vehicle': '#2563eb',
    'plane': '#f59e0b',
    'ship': '#06b6d4',
    'storage tank': '#8b5cf6',
    'bridge': '#ec4899',
    'harbor': '#14b8a6',
    'building': '#f97316',
    'buildings': '#f97316',
    'road / impervious': '#eab308',
    'tree': '#22c55e',
    'low vegetation': '#84cc16',
    'car': '#ef4444'
}};
const PALETTE = ['#ef4444','#2563eb','#10b981','#f59e0b','#8b5cf6','#ec4899','#06b6d4','#f97316','#84cc16','#6366f1'];

function getClassColor(label) {{
    if (CLASS_COLORS[label]) return CLASS_COLORS[label];
    let hash = 0;
    for (let i = 0; i < label.length; i++) hash = (hash * 31 + label.charCodeAt(i)) | 0;
    return PALETTE[Math.abs(hash) % PALETTE.length];
}}

const map = L.map('map').setView([25.0330, 121.5054], 15);
let currentTileLayer = null;

function setBasemap(key, flyToBounds = true) {{
    const cfg = BASEMAPS[key];
    if (!cfg) return;
    if (currentTileLayer) map.removeLayer(currentTileLayer);
    currentTileLayer = L.tileLayer(cfg.url, {{ maxZoom: cfg.max_zoom, attribution: cfg.attribution }}).addTo(map);
    if (flyToBounds && cfg.bounds) {{
        map.fitBounds(cfg.bounds, {{ maxZoom: 17 }});
    }}
}}

const basemapSelect = document.getElementById('basemapSelect');
const basemapFilter = document.getElementById('basemapFilter');
function populateBasemaps(filterText = '') {{
    const q = filterText.trim().toLowerCase();
    const prev = basemapSelect.value || DEFAULT_BASEMAP;
    basemapSelect.innerHTML = '';
    Object.entries(BASEMAPS).forEach(([k, cfg]) => {{
        if (q && !cfg.name.toLowerCase().includes(q) && !k.toLowerCase().includes(q)) return;
        const opt = document.createElement('option');
        opt.value = k;
        opt.textContent = cfg.name;
        if (k === prev) opt.selected = true;
        basemapSelect.appendChild(opt);
    }});
}}
populateBasemaps();
basemapFilter.oninput = () => populateBasemaps(basemapFilter.value);
basemapSelect.onchange = () => setBasemap(basemapSelect.value, true);
setBasemap(DEFAULT_BASEMAP, false);

const modelSelect = document.getElementById('modelSelect');
Object.entries(MODELS).forEach(([k, cfg]) => {{
    const opt = document.createElement('option');
    opt.value = k;
    opt.textContent = cfg.available ? cfg.name : `${{cfg.name}} (upload weights)`;
    if (k === DEFAULT_MODEL) opt.selected = true;
    modelSelect.appendChild(opt);
}});

const confSlider = document.getElementById('confSlider');
const confVal = document.getElementById('confVal');
confSlider.oninput = () => {{ confVal.textContent = parseFloat(confSlider.value).toFixed(2); }};

const drawnItems = new L.FeatureGroup().addTo(map);
map.addControl(new L.Control.Draw({{
    draw: {{ rectangle: true, polygon: false, polyline: false, circle: false, marker: false, circlemarker: false }},
    edit: {{ featureGroup: drawnItems }}
}}));
map.on(L.Draw.Event.CREATED, e => {{
    drawnItems.clearLayers();
    drawnItems.addLayer(e.layer);
}});

const detectionLayer = L.layerGroup().addTo(map);
const statusEl = document.getElementById('status');
const legendEl = document.getElementById('legend');
let lastGeoJSON = null;

function updateLegend(counts) {{
    const entries = Object.entries(counts);
    if (entries.length === 0) {{ legendEl.innerHTML = ''; return; }}
    legendEl.innerHTML = entries
        .sort((a, b) => b[1] - a[1])
        .map(([label, cnt]) => `<div class="legend-item"><span class="swatch" style="background:${{getClassColor(label)}}"></span>${{label}} (${{cnt}})</div>`)
        .join('');
}}

document.getElementById('btnDetect').onclick = async () => {{
    if (drawnItems.getLayers().length === 0) {{ alert('Draw a rectangle first!'); return; }}
    const rect = drawnItems.getLayers()[0].getBounds();
    const payload = {{
        sw_lat: rect.getSouthWest().lat,
        sw_lon: rect.getSouthWest().lng,
        ne_lat: rect.getNorthEast().lat,
        ne_lon: rect.getNorthEast().lng,
        basemap: basemapSelect.value,
        model_id: modelSelect.value,
        conf: parseFloat(confSlider.value)
    }};
    detectionLayer.clearLayers();
    updateLegend({{}});
    statusEl.textContent = 'Running detection...';

    try {{
        const res = await fetch('/detect', {{
            method: 'POST',
            headers: {{ 'Content-Type': 'application/json' }},
            body: JSON.stringify(payload)
        }});
        const data = await res.json();
        if (!res.ok) {{
            statusEl.textContent = 'Error: ' + (data.detail || res.statusText);
            return;
        }}
        lastGeoJSON = data;
        const counts = {{}};
        data.features.forEach(f => {{
            const coords = f.geometry.coordinates[0].map(c => [c[1], c[0]]);
            const label = f.properties.label;
            counts[label] = (counts[label] || 0) + 1;
            const color = getClassColor(label);
            L.polygon(coords, {{ color: color, weight: 1.5, fillOpacity: 0.2 }})
                .bindTooltip(`${{label}} (${{f.properties.score.toFixed(2)}})`) 
                .addTo(detectionLayer);
        }});
        statusEl.textContent = `Detected ${{data.features.length}} object(s).`;
        updateLegend(counts);
    }} catch (err) {{
        statusEl.textContent = 'Request failed: ' + err.message;
    }}
}};

document.getElementById('btnDownload').onclick = () => {{
    if (!lastGeoJSON) {{ alert('No detection results to download!'); return; }}
    const blob = new Blob([JSON.stringify(lastGeoJSON, null, 2)], {{ type: 'application/json' }});
    const url = URL.createObjectURL(blob);
    const a = document.createElement('a');
    a.href = url;
    a.download = `detection_${{modelSelect.value}}.geojson`;
    a.click();
    URL.revokeObjectURL(url);
}};

document.getElementById('btnClear').onclick = () => {{
    detectionLayer.clearLayers();
    lastGeoJSON = null;
    updateLegend({{}});
    statusEl.textContent = 'Detections cleared.';
}};
</script>
</body>
</html>
"""
    return HTMLResponse(content=html)

public_url = ngrok.connect(8000)
print("Public URL:", public_url)

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)
await server.serve()
